# Semi-Supervised MNIST Classification — Full Pipeline (v2)

**Flow:**
- Stage 1: SimCLR (Modified ResNet-18) → Label Propagation (+ confidence filter)
- Stage 2: cVAE pretraining (β annealing)
- Stage 3: GAN main loop (frozen SimCLR + EfficientNet features projected → concat, trainable ResNet + SpectralNorm)
- Inference: SWA-D + TTA (5 aug views → softmax avg)

**改善点 (v2):**
- 全フェーズで train/val 分割 (labeled 500 → 400 train / 100 val、stratified)
- test 10000件は最後の1回のみ使用
- LP: lp_alpha 0.99→0.92、Top-K per class + 信頼度フロア (curriculum動的閾値)
- SimCLR: Gradient Accumulation (実効バッチ ×4) + val kNN acc
- FixMatch式 Weak/Strong分離: 疑似ラベル生成は weak aug、学習は strong aug
- Stage 3毎エポック: train loss / val acc をログ

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import spectral_norm
from torch.utils.data import Dataset, DataLoader, Subset, TensorDataset
from torch.optim.swa_utils import AveragedModel, SWALR
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torchvision.models as tvm
import numpy as np
import random
import math
import copy
import os
from dataclasses import dataclass
from typing import Optional, Tuple, List
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


Device: cuda


In [2]:
@dataclass
class Config:
    # Data
    n_labeled:        int   = 500
    n_unlabeled:      int   = 3000
    n_classes:        int   = 10
    img_size:         int   = 32
    val_ratio:        float = 0.2   # labeled を 80/20 で train/val 分割

    # Stage 1: SimCLR
    simclr_epochs:    int   = 200
    simclr_lr:        float = 3e-4
    simclr_batch:     int   = 256
    simclr_accum:     int   = 4     # gradient accumulation → 実効バッチ 1024
    simclr_feat:      int   = 256   # WRN-28-4: 64*4=256
    simclr_proj:      int   = 128
    simclr_temp:      float = 0.5

    # WideResNet
    wrn_depth:        int   = 28
    wrn_width:        int   = 4          # feat_dim = 64 * width = 256

    # Stage 1b: Label Propagation
    lp_k:             int   = 15
    lp_alpha:         float = 0.92   # 0.99→0.92: ラベルの影響力を残す
    lp_iters:         int   = 60
    # Dynamic threshold: Top-K per class (カリキュラム方式)
    lp_topk_init:     int   = 20    # 初回: 各クラス上位20件
    lp_topk_max:      int   = 50    # 最大: 各クラス上位50件
    lp_conf_floor:    float = 0.70  # Top-Kでも最低この信頼度は必要

    # Stage 2: cVAE
    cvae_epochs:      int   = 100
    cvae_lr:          float = 1e-3
    cvae_batch:       int   = 128
    latent_dim:       int   = 64
    beta_start:       float = 1.0   # 4.0→1.0: KL支配でグレー崩壊するのを防ぐ
    beta_end:         float = 0.5
    lambda_percep:    float = 0.1

    # Stage 3: GAN
    gan_epochs:       int   = 300
    gan_lr_d:         float = 2e-4
    gan_lr_g:         float = 2e-4
    gan_lr_cls:       float = 1e-4
    gan_batch:        int   = 128
    fixmatch_thresh: float = 0.95
    fixmatch_temp:   float = 0.5

    # Loss weights
    lam_cls:          float = 1.0
    lam_gen:          float = 0.5
    lam_sc:           float = 0.3
    lam_pseudo:       float = 0.5
    lam_fm:           float = 1.0

    warmup_gen:       int   = 20
    warmup_sc:        int   = 30
    pseudo_refresh:   int   = 20   # epoch間隔でLP再計算 & topk増加

    # EMA / SWA
    ema_decay:        float = 0.999
    swa_start:        float = 0.75

    # ADA
    ada_target:       float = 0.6
    ada_interval:     int   = 4

    # Feature dims
    simclr_comp:      int   = 128
    eff_comp:         int   = 128
    train_feat:       int   = 256   # WRN-28-4: 64*4=256
    disc_hidden:      int   = 512   # 128+128+256=512

    # Logging
    log_interval:     int   = 10   # epoch間隔でval acc表示

    device: str = DEVICE

cfg = Config()

In [3]:
class MNISTSemiDataset(Dataset):
    def __init__(self, images: torch.Tensor, labels: torch.Tensor):
        self.images = images
        self.labels = labels
    def __len__(self): return len(self.images)
    def __getitem__(self, i): return self.images[i], self.labels[i]

def load_mnist(cfg: Config):
    """
    MNIST を読み込み、labeled を stratified に train_l / val_l へ分割。
    test は返すが最後の評価にのみ使う。
    """
    tf = T.Compose([
        T.Resize(cfg.img_size),
        T.ToTensor(),
        T.Lambda(lambda x: x.repeat(3, 1, 1)),
    ])
    full_train = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=tf)
    test_ds    = torchvision.datasets.MNIST("./data", train=False, download=True, transform=tf)

    per_class = cfg.n_labeled // cfg.n_classes
    labeled_idx, unlabeled_idx = [], []
    counts = {c: 0 for c in range(cfg.n_classes)}
    perm = torch.randperm(len(full_train)).tolist()
    for i in perm:
        _, lbl = full_train[i]
        if counts[lbl] < per_class:
            labeled_idx.append(i); counts[lbl] += 1
        elif len(unlabeled_idx) < cfg.n_unlabeled:
            unlabeled_idx.append(i)
        if len(labeled_idx) == cfg.n_labeled and len(unlabeled_idx) == cfg.n_unlabeled:
            break

    def collect(ds, idx, mask_label=False):
        imgs, lbls = [], []
        for i in idx:
            im, lb = ds[i]
            imgs.append(im)
            lbls.append(-1 if mask_label else lb)
        return torch.stack(imgs), torch.tensor(lbls)

    lx, ly = collect(full_train, labeled_idx)
    ux, _  = collect(full_train, unlabeled_idx, mask_label=True)

    # --- Stratified train/val split of labeled data ---
    val_per_class = max(1, int(per_class * cfg.val_ratio))
    train_idx, val_idx = [], []
    vcounts = {c: 0 for c in range(cfg.n_classes)}
    # shuffle within labeled
    lperm = torch.randperm(len(lx)).tolist()
    for i in lperm:
        c = ly[i].item()
        if vcounts[c] < val_per_class:
            val_idx.append(i); vcounts[c] += 1
        else:
            train_idx.append(i)

    lx_train, ly_train = lx[train_idx], ly[train_idx]
    lx_val,   ly_val   = lx[val_idx],   ly[val_idx]

    # test
    test_imgs = torch.stack([test_ds[i][0] for i in range(len(test_ds))])
    test_lbls = torch.tensor([test_ds[i][1] for i in range(len(test_ds))])

    print(f"Labeled train: {len(lx_train)}, Labeled val: {len(lx_val)}, "
          f"Unlabeled: {len(ux)}, Test: {len(test_imgs)}")
    return lx_train, ly_train, lx_val, ly_val, ux, test_imgs, test_lbls

lx_train, ly_train, lx_val, ly_val, ux, test_imgs, test_lbls = load_mnist(cfg)

Labeled train: 400, Labeled val: 100, Unlabeled: 3000, Test: 10000


In [4]:
#
# FixMatch 式の分離:
#   weak   : crop + hflip のみ → 疑似ラベルの「決定」に使用
#   strong : DiffAugment (color/translation/cutout) → 学習時に使用
#   disc   : ADA確率で strong を適用 → D の real/fake に使用
#   supcon : strong x2 → SupCon ペア
#   cls    : curriculum (epoch に応じて weak → weak+Mixup/CutMix)

# ---------- DiffAugment ----------
def rand_rotation(x, max_deg=15):
    angles = [random.uniform(-max_deg, max_deg) for _ in range(x.size(0))]
    return torch.stack([TF.rotate(xi, a) for xi, a in zip(x, angles)])
def rand_brightness(x):
    f = torch.empty(x.size(0),1,1,1,device=x.device).uniform_(0.5,1.5)
    return (x*f).clamp(0,1)
def rand_saturation(x):
    g = x.mean(dim=1,keepdim=True)
    f = torch.empty(x.size(0),1,1,1,device=x.device).uniform_(0.5,1.5)
    return torch.lerp(g,x,f).clamp(0,1)
def rand_contrast(x):
    m = x.mean(dim=(1,2,3),keepdim=True)
    f = torch.empty(x.size(0),1,1,1,device=x.device).uniform_(0.5,1.5)
    return torch.lerp(m,x,f).clamp(0,1)
def rand_translation(x, ratio=0.125):
    B,C,H,W = x.shape
    sh=int(H*ratio); sw=int(W*ratio)
    tx=torch.randint(-sw,sw+1,(B,)); ty=torch.randint(-sh,sh+1,(B,))
    theta=torch.zeros(B,2,3,device=x.device)
    theta[:,0,0]=1; theta[:,1,1]=1
    theta[:,0,2]=tx.float()/W*2; theta[:,1,2]=ty.float()/H*2
    grid=F.affine_grid(theta,x.shape,align_corners=False)
    return F.grid_sample(x,grid,align_corners=False,padding_mode="reflection")
def rand_cutout(x, ratio=0.5):
    B,C,H,W=x.shape; ch=int(H*ratio); cw=int(W*ratio)
    out=x.clone()
    for i in range(B):
        t=random.randint(0,H-ch); l=random.randint(0,W-cw)
        out[i,:,t:t+ch,l:l+cw]=0
    return out

def diff_augment(x, policy="color,translation,cutout,rotation"):
    for p in policy.split(","):
        p=p.strip()
        if p=="color":
            for fn in [rand_brightness,rand_saturation,rand_contrast]: x=fn(x)
        elif p=="translation": x=rand_translation(x)
        elif p=="cutout":      x=rand_cutout(x)
        elif p == "rotation": x = rand_rotation(x)
    return x.clamp(0,1)

# ---------- Weak augment (FixMatch 決定用) ----------
_weak_tf = T.Compose([T.RandomCrop(32,padding=4), T.RandomHorizontalFlip()])
def weak_aug(x: torch.Tensor) -> torch.Tensor:
    return torch.stack([_weak_tf(xi) for xi in x])

# ---------- Mixup / CutMix ----------
def mixup(x,y,alpha=0.4):
    lam=np.random.beta(alpha,alpha)
    idx=torch.randperm(x.size(0),device=x.device)
    return lam*x+(1-lam)*x[idx], y, y[idx], lam

def cutmix(x,y,alpha=1.0):
    lam=np.random.beta(alpha,alpha)
    B,C,H,W=x.shape
    cr=math.sqrt(1-lam); ch=int(H*cr); cw=int(W*cr)
    cx=random.randint(0,W); cy=random.randint(0,H)
    x1=max(0,cx-cw//2); x2=min(W,cx+cw//2)
    y1=max(0,cy-ch//2); y2=min(H,cy+ch//2)
    lam=1-(x2-x1)*(y2-y1)/(W*H)
    idx=torch.randperm(B,device=x.device)
    out=x.clone(); out[:,:,y1:y2,x1:x2]=x[idx,:,y1:y2,x1:x2]
    return out,y,y[idx],lam

def mix_criterion(crit,pred,ya,yb,lam):
    return lam*crit(pred,ya)+(1-lam)*crit(pred,yb)

# ---------- ADA ----------
class ADAController:
    def __init__(self, target=0.6, interval=4, speed=1e-3):
        self.p=0.0; self.target=target
        self.interval=interval; self.speed=speed
        self._step=0; self._signs=[]
    def update(self, real_logits):
        self._signs.append((real_logits>0).float().mean().item())
        self._step+=1
        if self._step%self.interval==0:
            rt=np.mean(self._signs[-self.interval:])
            adj=np.sign(rt-self.target)*self.speed*self.interval
            self.p=float(np.clip(self.p+adj,0,1))
    def apply(self, x):
        return diff_augment(x) if random.random()<self.p else x

ada = ADAController(target=cfg.ada_target, interval=cfg.ada_interval)

# ---------- Curriculum classifier view ----------
def get_cls_view(x, y, epoch, cfg):
    """
    Curriculum:
      < warmup_gen      : weak のみ
      warmup_gen〜60    : weak
      60〜120           : weak + 50% Mixup
      120〜             : weak + 50% (Mixup or CutMix)
    疑似ラベル決定には使わない (weak_aug を別途使う)
    """
    x = weak_aug(x)
    if epoch < 60:
        return x, y, y, 1.0
    if epoch < 120:
        if random.random() < 0.5:
            return mixup(x, y)
        return x, y, y, 1.0
    if random.random() < 0.5:
        fn = mixup if random.random()<0.5 else cutmix
        return fn(x, y)
    return x, y, y, 1.0

In [5]:
def sn_linear(i, o, **kw): return spectral_norm(nn.Linear(i, o, **kw))

# ================================================================
# WideResNet (CIFAR スタイル 32x32 対応)
# WRN-28-4: depth=28, width=4 → feat_dim = 64*4 = 256
# ================================================================
class WideResNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.0, use_sn=False):
        super().__init__()
        W = spectral_norm if use_sn else (lambda m: m)
        self.bn1   = nn.BatchNorm2d(in_ch)
        self.conv1 = W(nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False))
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.drop  = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.conv2 = W(nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False))
        self.skip  = (W(nn.Conv2d(in_ch, out_ch, 1, stride, bias=False))
                      if stride != 1 or in_ch != out_ch else nn.Identity())

    def forward(self, x):
        out = self.conv1(F.relu(self.bn1(x), inplace=True))
        out = self.conv2(F.relu(self.bn2(self.drop(out)), inplace=True))
        return out + self.skip(x)


class WideResNet(nn.Module):
    """
    WRN-depth-width for 32x32 inputs.
      depth=28, width=4, dropout=0.0  → SimCLR backbone
      depth=28, width=4, dropout=0.3, use_sn=True → D trainable backbone
    feat_dim = 64 * width  (WRN-28-4: 256)
    """
    def __init__(self, depth=28, width=4, dropout=0.0, use_sn=False):
        super().__init__()
        assert (depth - 4) % 6 == 0, "depth must satisfy (depth-4)%6==0"
        n  = (depth - 4) // 6
        k  = width
        ch = [16, 16*k, 32*k, 64*k]
        W  = spectral_norm if use_sn else (lambda m: m)
        self.conv0  = W(nn.Conv2d(3, ch[0], 3, 1, 1, bias=False))
        self.layer1 = self._make(n, ch[0], ch[1], 1, dropout, use_sn)
        self.layer2 = self._make(n, ch[1], ch[2], 2, dropout, use_sn)
        self.layer3 = self._make(n, ch[2], ch[3], 2, dropout, use_sn)
        self.bn     = nn.BatchNorm2d(ch[3])
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.feat_dim = ch[3]  # = 64 * width

    def _make(self, n, ic, oc, stride, dropout, use_sn):
        layers = [WideResNetBlock(ic, oc, stride, dropout, use_sn)]
        for _ in range(n - 1):
            layers.append(WideResNetBlock(oc, oc, 1, dropout, use_sn))
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv0(x)
        out = self.layer3(self.layer2(self.layer1(out)))
        return self.pool(F.relu(self.bn(out), inplace=True)).flatten(1)


class SimCLR(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone  = WideResNet(cfg.wrn_depth, cfg.wrn_width, dropout=0.0)
        feat = self.backbone.feat_dim  # 256 for WRN-28-4
        self.projector = nn.Sequential(
            nn.Linear(feat, feat), nn.ReLU(True),
            nn.Linear(feat, cfg.simclr_proj))

    def encode(self, x): return self.backbone(x)

    def forward(self, x):
        return F.normalize(self.projector(self.backbone(x)), dim=-1)


class EfficientNetB0Features(nn.Module):
    def __init__(self):
        super().__init__()
        eff = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.DEFAULT)
        self.features  = eff.features
        self.pool      = eff.avgpool
        self.normalize = T.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])

    def forward(self, x):
        return self.pool(self.features(self.normalize(x))).flatten(1)


class CVAEEncoder(nn.Module):
    def __init__(self, img_ch=3, img_size=32, n_classes=10, latent_dim=64):
        super().__init__()
        self.embed = nn.Embedding(n_classes, 16)
        self.net   = nn.Sequential(
            nn.Conv2d(img_ch+16, 64,  4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64,  128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, True),
            nn.Flatten())
        self.fc_mu     = nn.Linear(256*4*4, latent_dim)
        self.fc_logvar = nn.Linear(256*4*4, latent_dim)

    def forward(self, x, y):
        e = self.embed(y)[:, :, None, None].expand(-1, -1, x.size(2), x.size(3))
        h = self.net(torch.cat([x, e], 1))
        return self.fc_mu(h), self.fc_logvar(h)


class Generator(nn.Module):
    """
    グレー背景修正:
      1. 最終活性化: Tanh → Sigmoid  (出力 [0,1], (out+1)/2 不要)
      2. 最終 Conv bias を +1.5 初期化 → sigmoid(1.5)≈0.82 (白寄りスタート)
      3. 全 ConvTranspose2d に SpectralNorm
    """
    def __init__(self, latent_dim=64, n_classes=10, img_ch=3, img_size=32):
        super().__init__()
        self.latent_dim = latent_dim
        self.img_size   = img_size
        self.embed = nn.Embedding(n_classes, 64)
        self.fc    = nn.Linear(latent_dim + 64, 256 * 4 * 4)
        final_conv = nn.Conv2d(32, img_ch, 3, 1, 1)
        nn.init.constant_(final_conv.bias, 1.5)  # sigmoid(1.5)≈0.82
        self.net = nn.Sequential(
            nn.BatchNorm2d(256),
            spectral_norm(nn.ConvTranspose2d(256, 128, 4, 2, 1)),
            nn.BatchNorm2d(128), nn.ReLU(True),
            spectral_norm(nn.ConvTranspose2d(128, 64, 4, 2, 1)),
            nn.BatchNorm2d(64),  nn.ReLU(True),
            spectral_norm(nn.ConvTranspose2d(64, 32, 4, 2, 1)),
            nn.BatchNorm2d(32),  nn.ReLU(True),
            spectral_norm(final_conv),
            nn.Sigmoid(),  # ← Tanh から変更; 出力は直接 [0,1]
        )

    def forward(self, z, y):
        e   = self.embed(y)
        h   = F.relu(self.fc(torch.cat([z, e], 1)), inplace=True).view(-1, 256, 4, 4)
        out = self.net(h)
        if out.size(-1) != self.img_size:
            out = F.interpolate(out, self.img_size, mode="bilinear", align_corners=False)
        return out  # [0,1]; (out+1)/2 不要


class Discriminator(nn.Module):
    """
    入力特徴: 凍結 SimCLR WRN-28-4 (256→comp128)
            + 凍結 EfficientNet-B0 (1280→comp128)
            + 訓練可能 WRN-28-4 (256, SN+dropout=0.3)
    合計: 128+128+256 = 512 = disc_hidden
    """
    def __init__(self, cfg, simclr_backbone, eff_backbone):
        super().__init__()
        self.simclr_backbone = simclr_backbone
        self.eff_backbone    = eff_backbone
        for p in self.simclr_backbone.parameters(): p.requires_grad_(False)
        for p in self.eff_backbone.parameters():    p.requires_grad_(False)

        self.simclr_comp = nn.Sequential(
            sn_linear(cfg.simclr_feat, cfg.simclr_comp), nn.LeakyReLU(0.2, True))
        self.eff_comp    = nn.Sequential(
            sn_linear(1280, cfg.eff_comp), nn.LeakyReLU(0.2, True))

        # 訓練可能バックボーン: WRN-28-4 + SpectralNorm + dropout=0.3
        self.train_backbone = WideResNet(cfg.wrn_depth, cfg.wrn_width,
                                         dropout=0.3, use_sn=True)

        total = cfg.simclr_comp + cfg.eff_comp + cfg.train_feat  # 128+128+256=512
        self.shared   = nn.Sequential(
            sn_linear(total, cfg.disc_hidden), nn.LeakyReLU(0.2, True),
            sn_linear(cfg.disc_hidden, cfg.simclr_proj))
        self.cls_head = sn_linear(cfg.simclr_proj, cfg.n_classes + 1)  # 11クラス
        self.embed    = nn.Embedding(cfg.n_classes, cfg.simclr_proj)   # projection D

    def features(self, x):
        with torch.no_grad():
            fs = self.simclr_backbone(x)
            fe = self.eff_backbone(x)
        return self.shared(torch.cat(
            [self.simclr_comp(fs), self.eff_comp(fe), self.train_backbone(x)], 1))

    def forward(self, x, y=None):
        h      = self.features(x)
        logits = self.cls_head(h)
        if y is not None:
            proj = (h * self.embed(y)).sum(1)
            logits[torch.arange(len(y)), y] += proj
        return logits  # (B, n_classes+1)


In [6]:
class NTXentLoss(nn.Module):
    def __init__(self,temperature=0.5): super().__init__(); self.T=temperature
    def forward(self,z1,z2):
        N=z1.size(0); z=torch.cat([z1,z2],0)
        sim=F.cosine_similarity(z.unsqueeze(1),z.unsqueeze(0),dim=-1)/self.T
        mask=torch.eye(2*N,device=z.device).bool(); sim.masked_fill_(mask,-1e9)
        tgt=torch.arange(N,device=z.device); tgt=torch.cat([tgt+N,tgt])
        return F.cross_entropy(sim,tgt)

class SupConLoss(nn.Module):
    def __init__(self,temperature=0.07): super().__init__(); self.T=temperature
    def forward(self,features,labels):
        B=features.size(0); f=F.normalize(features.view(2*B,-1),dim=-1)
        sim=torch.matmul(f,f.T)/self.T
        mask=torch.eye(2*B,device=f.device).bool(); sim.masked_fill_(mask,-1e9)
        lbl=labels.repeat_interleave(2)
        pos_mask=((lbl.unsqueeze(0)==lbl.unsqueeze(1))&~mask).float()
        log_prob=sim-torch.log(torch.exp(sim).sum(1,keepdim=True)+1e-8)
        loss=-(pos_mask*log_prob).sum(1)/(pos_mask.sum(1).clamp(min=1))
        return loss.mean()

class HingeLoss:
    @staticmethod
    def d_real(l): return F.relu(1.-l).mean()
    @staticmethod
    def d_fake(l): return F.relu(1.+l).mean()
    @staticmethod
    def g(l):      return -l.mean()

def feature_matching_loss(rf,ff): return F.l1_loss(ff,rf.detach())
def soft_kl_loss(logits,soft):
    return F.kl_div(F.log_softmax(logits,dim=-1),soft.clamp(1e-6),reduction="batchmean")
def vae_loss(recon,x,mu,logvar,beta,perc_fn=None,perc_w=0.):
    # BCE: MNIST は二値画像のため L1 よりグレー崩壊防止に有効
    rl=F.binary_cross_entropy(recon, x, reduction='mean')
    kl=-0.5*(1+logvar-mu.pow(2)-logvar.exp()).mean()
    loss=rl+beta*kl
    if perc_fn and perc_w>0: loss+=perc_w*perc_fn(recon,x)
    return loss,rl,kl

In [16]:
@torch.no_grad()
def knn_accuracy(model, db_imgs, db_lbls, query_imgs, query_lbls,
                 k=5, batch=256, device=DEVICE, loo=False):
    """
    kNN accuracy.
    loo=True: leave-one-out (db == query, self を除外)
    """
    model.eval()
    def extract(imgs):
        feats = []
        for i in range(0, len(imgs), batch):
            feats.append(model.encode(imgs[i:i+batch].to(device)).cpu())
        return torch.cat(feats, 0)
    f_db    = F.normalize(extract(db_imgs),    dim=-1)
    f_query = F.normalize(extract(query_imgs), dim=-1)
    sim = f_query @ f_db.T          # (N_query, N_db)
    if loo:
        sim.fill_diagonal_(-1e9)    # 自分自身を除外
    topk_idx  = sim.topk(k, dim=1).indices
    pred_lbls = db_lbls[topk_idx]
    preds, _  = pred_lbls.mode(dim=1)
    return (preds == query_lbls).float().mean().item()

@torch.no_grad()
def simclr_val_loss(model, lx_val, crit, device):
    """lx_val で NT-Xent loss を計算 (val loss)。"""
    model.eval()
    x  = lx_val.to(device)
    v1 = diff_augment(x); v2 = diff_augment(x)
    loss = crit(model(v1), model(v2)).item()
    model.train()
    return loss

def train_simclr(cfg, lx_train, lx_val, ly_train, ly_val, ux):
    """
    SimCLR 学習。毎 20 epoch で 4 指標を表示:
      train_loss / val_loss / train_kNN_acc (LOO) / val_kNN_acc
    """
    all_imgs = torch.cat([lx_train, ux], 0)
    ds       = TensorDataset(all_imgs)
    loader   = DataLoader(ds, batch_size=cfg.simclr_batch, shuffle=True,
                          num_workers=0, pin_memory=True, drop_last=True)

    model = SimCLR(cfg).to(cfg.device)
    opt   = torch.optim.Adam(model.parameters(), lr=cfg.simclr_lr, weight_decay=1e-6)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, cfg.simclr_epochs)
    crit  = NTXentLoss(cfg.simclr_temp)

    os.makedirs("models/sgan", exist_ok=True)
    save_path = "models/sgan/simclr_best.pth"
    if os.path.exists(save_path):
        tqdm.write(f"Loading SimCLR from {save_path}")
        model.load_state_dict(torch.load(save_path, map_location=cfg.device, weights_only=True))
        return model

    best_val_acc = 0.0
    A = cfg.simclr_accum  # gradient accumulation steps

    for epoch in range(cfg.simclr_epochs):
        model.train(); total = 0.0; opt.zero_grad()

        step_bar = tqdm(loader, desc=f"Epoch {epoch+1}", position=0, leave=True, unit="batch")
        for step, (x,) in enumerate(step_bar):
            x = x.to(cfg.device)
            v1 = diff_augment(x); v2 = diff_augment(x)
            loss = crit(model(v1), model(v2)) / A
            loss.backward()
            if (step + 1) % A == 0:
                opt.step(); opt.zero_grad()
            total += loss.item() * A

            step_bar.set_postfix(loss=f"{loss.item()*A:.4f}")

        # 端数ステップを確実に flush
        if len(loader) % A != 0:
            opt.step(); opt.zero_grad()

        sched.step()
        avg_train_loss = total / len(loader)

        if (epoch + 1) % 20 == 0:
            val_loss   = simclr_val_loss(model, lx_val, crit, cfg.device)
            train_kacc = knn_accuracy(model, lx_train, ly_train,
                                      lx_train, ly_train, loo=True, device=cfg.device)
            val_kacc   = knn_accuracy(model, lx_train, ly_train,
                                      lx_val,   ly_val,   device=cfg.device)
            tqdm.write(
                f"[SimCLR ep {epoch+1:03d}/{cfg.simclr_epochs}] "
                f"train_loss={avg_train_loss:.4f}  val_loss={val_loss:.4f} | "
                f"train_kNN={train_kacc*100:.1f}%  val_kNN={val_kacc*100:.1f}%"
            )
            if val_kacc > best_val_acc:
                best_val_acc = val_kacc
                torch.save(model.state_dict(), save_path)
                tqdm.write(f"  => [SimCLR] Best val kNN={best_val_acc*100:.1f}%  saved.")

    model.load_state_dict(torch.load(save_path, map_location=cfg.device, weights_only=True))
    tqdm.write(f"[SimCLR] Best val kNN acc: {best_val_acc*100:.1f}%")
    return model

print("=== Stage 1: SimCLR ===")
simclr_model = train_simclr(cfg, lx_train, lx_val, ly_train, ly_val, ux)


=== Stage 1: SimCLR ===


Epoch 20: 100%|██████████| 13/13 [00:13<00:00,  1.01s/batch, loss=5.1320]


[SimCLR ep 020/200] train_loss=5.1078  val_loss=4.1784 | train_kNN=58.5%  val_kNN=60.0%
  => [SimCLR] Best val kNN=60.0%  saved.


Epoch 40: 100%|██████████| 13/13 [00:11<00:00,  1.15batch/s, loss=4.8654]


[SimCLR ep 040/200] train_loss=4.8961  val_loss=4.0679 | train_kNN=74.8%  val_kNN=69.0%
  => [SimCLR] Best val kNN=69.0%  saved.


...

[SimCLR ep 160/200] train_loss=4.6013  val_loss=3.6854 | train_kNN=91.8%  val_kNN=91.0%


Epoch 180: 100%|██████████| 13/13 [00:11<00:00,  1.10batch/s, loss=4.5950]


[SimCLR ep 180/200] train_loss=4.5887  val_loss=3.6531 | train_kNN=92.0%  val_kNN=93.0%
  => [SimCLR] Best val kNN=93.0%  saved.


Epoch 200: 100%|██████████| 13/13 [00:11<00:00,  1.11batch/s, loss=4.5814]


[SimCLR ep 200/200] train_loss=4.5833  val_loss=3.6610 | train_kNN=92.3%  val_kNN=92.0%
[SimCLR] Best val kNN acc: 93.0%


In [17]:
def label_propagation(cfg, simclr_model, lx_train, ly_train, ux,
                      refresh_count: int = 0, D_model=None):
    """
    refresh_count: 何回目の LP か (0=初回)。topk をカリキュラム的に増やす。
    動的閾値:
      - 各クラスの信頼スコア上位 topk 件を選択
      - ただし conf < lp_conf_floor のものは除外
    """
    from scipy.sparse import csr_matrix

    all_imgs=torch.cat([lx_train,ux],0).to(cfg.device)
    N_l=len(lx_train); N=len(all_imgs)
    n_refreshes=cfg.gan_epochs//cfg.pseudo_refresh

    # Curriculum topk: 初回→最終でlinear補間
    frac=min(1.0, refresh_count/(max(1,n_refreshes)))
    topk=int(cfg.lp_topk_init+(cfg.lp_topk_max-cfg.lp_topk_init)*frac)
    print(f"  [LP] refresh#{refresh_count}  α={cfg.lp_alpha}  topk={topk}/class  "
          f"conf_floor={cfg.lp_conf_floor}")

    simclr_model.eval()
    with torch.no_grad():
        feats=[]
        for i in range(0,N,256):
            feats.append(simclr_model.encode(all_imgs[i:i+256]))
        feats=torch.cat(feats,0).cpu().numpy()
    feats/=(np.linalg.norm(feats,axis=1,keepdims=True)+1e-8)

    # 初回はSimCLR only、refresh_count >= 2 以降はアンサンブル
    if refresh_count >= 2 and D_model is not None:
        D_model.eval()
        with torch.no_grad():
            d_feats = []
            for i in range(0, N, 256):
                d_feats.append(D_model.features(all_imgs[i:i+256]))
            d_feats = torch.cat(d_feats, 0).cpu().numpy()
            d_feats /= np.linalg.norm(d_feats, axis=1, keepdims=True) + 1e-8
        feats = np.concatenate([feats, d_feats], axis=1)  # 特徴を連結してcos類似度
        feats /= (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-8)

    # k-NN affinity
    sim=feats@feats.T; np.fill_diagonal(sim,-1)
    knn_idx=np.argsort(-sim,axis=1)[:,:cfg.lp_k]
    knn_val=np.exp(sim[np.arange(N)[:,None],knn_idx])
    rows=np.repeat(np.arange(N),cfg.lp_k); cols=knn_idx.ravel(); vals=knn_val.ravel()
    W=csr_matrix((vals,(rows,cols)),shape=(N,N)); W=(W+W.T)/2
    d=np.asarray(W.sum(1)).ravel()**-0.5
    D_mat=csr_matrix((d,(np.arange(N),np.arange(N))),shape=(N,N))
    S=(D_mat@W@D_mat).toarray().astype(np.float32)

    # Label matrix: labeled train のみ初期化
    Y=np.zeros((N,cfg.n_classes),dtype=np.float32)
    for i,lbl in enumerate(ly_train.numpy()):
        Y[i,lbl]=1.0

    F_mat=Y.copy()
    for _ in range(cfg.lp_iters):
        F_mat=cfg.lp_alpha*(S@F_mat)+(1-cfg.lp_alpha)*Y
    F_mat/=(F_mat.sum(1,keepdims=True)+1e-8)

    conf=F_mat.max(axis=1)           # (N,)
    hard=F_mat.argmax(axis=1)        # (N,)

    # --- Top-K per class selection (unlabeled 部分のみ) ---
    ul_F=F_mat[N_l:]                 # (N_u, n_classes)
    ul_conf=conf[N_l:]
    ul_hard=hard[N_l:]
    confident_ul=np.zeros(len(ux),dtype=bool)

    for c in range(cfg.n_classes):
        cls_idx = np.where(ul_hard == c)[0]  # クラスcと予測されたインデックスのみ
        if len(cls_idx) == 0: continue
        # そのクラス内で信頼度上位topk件を選択
        sorted_idx = cls_idx[np.argsort(-ul_conf[cls_idx])]
        for i in sorted_idx[:topk]:
            if ul_conf[i] >= cfg.lp_conf_floor:
                confident_ul[i] = True

    soft_all=torch.tensor(F_mat,dtype=torch.float32)
    # labeled部分は全信頼
    confident_mask=np.concatenate([np.ones(N_l,dtype=bool),confident_ul])

    n_conf_ul=confident_ul.sum()
    print(f"  [LP] confident unlabeled: {n_conf_ul}/{len(ux)}  "
          f"total (incl. labeled): {confident_mask.sum()}/{N}")
    return soft_all, confident_mask, hard

print("\n=== Stage 1b: Label Propagation ===")
soft_labels,lp_mask,lp_hard=label_propagation(cfg,simclr_model,lx_train,ly_train,ux,refresh_count=0)


=== Stage 1b: Label Propagation ===
  [LP] refresh#0  α=0.92  topk=20/class  conf_floor=0.7
  [LP] confident unlabeled: 200/3000  total (incl. labeled): 600/3400


In [18]:
@torch.no_grad()
def cvae_val_metrics(enc, gen, lx_val, ly_val, beta, simclr_ref, cfg):
    """
    val セットで total_loss と recon_loss の両方を返す。
    perceptual 損失の計算も含む。
    """
    enc.eval(); gen.eval(); simclr_ref.eval()
    x, y = lx_val.to(cfg.device), ly_val.to(cfg.device)
    mu, logvar = enc(x, y)
    z     = mu + (0.5 * logvar).exp() * torch.randn_like(mu)
    recon = gen(z, y)
    with torch.no_grad():
        fr = simclr_ref.encode(x)
        fp = simclr_ref.encode(recon)
    perc = F.l1_loss(fp, fr)
    rl   = F.binary_cross_entropy(recon, x, reduction='mean')  # BCEに統一
    kl   = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).mean()
    total = rl + beta * kl + cfg.lambda_percep * perc
    enc.train(); gen.train()
    return total.item(), rl.item()

def train_cvae(cfg, simclr_model, lx_train, ly_train, lx_val, ly_val):
    """
    cVAE 学習。毎 20 epoch で 4 指標を表示:
      train_loss / val_loss / train_recon / val_recon
    """
    ds     = TensorDataset(lx_train, ly_train)
    loader = DataLoader(ds, batch_size=cfg.cvae_batch, shuffle=True, drop_last=True)

    enc = CVAEEncoder(3, cfg.img_size, cfg.n_classes, cfg.latent_dim).to(cfg.device)
    gen = Generator(cfg.latent_dim, cfg.n_classes, 3, cfg.img_size).to(cfg.device)
    opt   = torch.optim.Adam(list(enc.parameters()) + list(gen.parameters()), lr=cfg.cvae_lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, cfg.cvae_epochs)
    simclr_model.eval()

    def perceptual(recon, real):
        with torch.no_grad():
            fr = simclr_model.encode(real)
            fp = simclr_model.encode(recon)
        return F.l1_loss(fp, fr)

    os.makedirs("models/sgan", exist_ok=True)
    sp_enc, sp_gen = "models/sgan/cvae_enc_best.pth", "models/sgan/cvae_gen_best.pth"
    if os.path.exists(sp_enc) and os.path.exists(sp_gen):
        tqdm.write("Loading cVAE from checkpoint")
        enc.load_state_dict(torch.load(sp_enc, map_location=cfg.device, weights_only=True))
        gen.load_state_dict(torch.load(sp_gen, map_location=cfg.device, weights_only=True))
        return enc, gen

    best_vl = float("inf")

    for epoch in range(cfg.cvae_epochs):
        beta = cfg.beta_start + (cfg.beta_end - cfg.beta_start) * epoch / cfg.cvae_epochs
        enc.train(); gen.train()
        total_loss = 0.; total_recon = 0.

        step_bar = tqdm(loader, desc=f"Epoch {epoch+1}", position=0, leave=True, unit="batch")
        for x, y in step_bar:
            x, y = x.to(cfg.device), y.to(cfg.device)
            mu, logvar = enc(x, y)
            z     = mu + torch.randn_like(mu) * (0.5 * logvar).exp()
            recon = gen(z, y)
            loss, rl, kl = vae_loss(recon, x, mu, logvar, beta,
                                    perc_fn=perceptual, perc_w=cfg.lambda_percep)
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss  += loss.item()
            total_recon += rl.item()

            step_bar.set_postfix(loss=f"{loss.item():.4f}")

        sched.step()
        avg_train_loss  = total_loss  / len(loader)
        avg_train_recon = total_recon / len(loader)

        if (epoch + 1) % 20 == 0:
            val_total, val_recon = cvae_val_metrics(
                enc, gen, lx_val, ly_val, beta, simclr_model, cfg)
            tqdm.write(
                f"[cVAE ep {epoch+1:03d}/{cfg.cvae_epochs}] "
                f"train_loss={avg_train_loss:.4f}  val_loss={val_total:.4f} | "
                f"train_recon={avg_train_recon:.4f}  val_recon={val_recon:.4f}  β={beta:.3f}"
            )
            if val_total < best_vl:
                best_vl = val_total
                torch.save(enc.state_dict(), sp_enc)
                torch.save(gen.state_dict(), sp_gen)

    enc.load_state_dict(torch.load(sp_enc, map_location=cfg.device, weights_only=True))
    gen.load_state_dict(torch.load(sp_gen, map_location=cfg.device, weights_only=True))
    return enc, gen

print("\n=== Stage 2: cVAE ===")
cvae_enc, cvae_gen = train_cvae(cfg, simclr_model, lx_train, ly_train, lx_val, ly_val)
G = copy.deepcopy(cvae_gen)



=== Stage 2: cVAE ===


Epoch 20: 100%|██████████| 3/3 [00:00<00:00,  8.78batch/s, loss=0.2770]


[cVAE ep 020/100] train_loss=0.2777  val_loss=0.2824 | train_recon=0.2405  val_recon=0.2450  β=0.905


Epoch 40: 100%|██████████| 3/3 [00:00<00:00,  8.49batch/s, loss=0.2748]


[cVAE ep 040/100] train_loss=0.2704  val_loss=0.2737 | train_recon=0.2334  val_recon=0.2365  β=0.805


...

[cVAE ep 060/100] train_loss=0.2657  val_loss=0.2699 | train_recon=0.2294  val_recon=0.2342  β=0.705


Epoch 80: 100%|██████████| 3/3 [00:00<00:00,  8.31batch/s, loss=0.2710]


[cVAE ep 080/100] train_loss=0.2656  val_loss=0.2682 | train_recon=0.2297  val_recon=0.2324  β=0.605


Epoch 100: 100%|██████████| 3/3 [00:00<00:00,  8.34batch/s, loss=0.2671]


[cVAE ep 100/100] train_loss=0.2647  val_loss=0.2675 | train_recon=0.2287  val_recon=0.2319  β=0.505


In [ ]:
eff_model = EfficientNetB0Features().to(cfg.device)
D = Discriminator(cfg, simclr_model.backbone, eff_model).to(cfg.device)
G = G.to(cfg.device)
G_ema = copy.deepcopy(G)

def update_ema(src, tgt, decay):
    with torch.no_grad():
        for ps, pt in zip(src.parameters(), tgt.parameters()):
            pt.data.mul_(decay).add_(ps.data, alpha=1 - decay)

D_swa = AveragedModel(D)

opt_D   = torch.optim.AdamW([p for p in D.parameters() if p.requires_grad],
                              lr=cfg.gan_lr_d, betas=(0., 0.99))
opt_G   = torch.optim.AdamW(G.parameters(), lr=cfg.gan_lr_g, betas=(0., 0.99))
cls_params = (list(D.train_backbone.parameters()) + list(D.shared.parameters()) +
              list(D.cls_head.parameters()) + list(D.embed.parameters()) +
              list(D.simclr_comp.parameters()) + list(D.eff_comp.parameters()))
opt_cls = torch.optim.AdamW(cls_params, lr=cfg.gan_lr_cls)

sched_D   = torch.optim.lr_scheduler.CosineAnnealingLR(opt_D,   cfg.gan_epochs)
sched_G   = torch.optim.lr_scheduler.CosineAnnealingLR(opt_G,   cfg.gan_epochs)
sched_cls = torch.optim.lr_scheduler.CosineAnnealingLR(opt_cls, cfg.gan_epochs)

hinge  = HingeLoss()
supcon = SupConLoss()
ce_fn  = nn.CrossEntropyLoss(label_smoothing=0.1)

def make_dataset1(lx_train, ly_train, ux, lp_mask, lp_hard):
    N_l    = len(lx_train)
    ul_mask = torch.tensor(lp_mask[N_l:])
    ul_idx  = torch.where(ul_mask)[0]
    imgs    = torch.cat([lx_train, ux[ul_idx]], 0)
    lbls    = torch.cat([ly_train, torch.tensor(lp_hard[N_l:][ul_idx.numpy()]).long()], 0)
    return TensorDataset(imgs, lbls)

def make_loader(ds, bs, shuffle=True):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle,
                      num_workers=0, pin_memory=True, drop_last=True)

@torch.no_grad()
def eval_metrics(model, lx, ly, cfg, batch=256):
    """
    lx/ly に対して:
      - 分類 loss (ce, label_smoothing なし)
      - 分類 accuracy
    を返す。train / val どちらにも使える。
    """
    ce_no_ls = torch.nn.CrossEntropyLoss()   # smoothing なし で純粋な loss
    model.eval()
    all_logits, preds_list = [], []
    for i in range(0, len(lx), batch):
        logits = model(lx[i:i+batch].to(cfg.device))[:, :cfg.n_classes]
        all_logits.append(logits.cpu())
        preds_list.append(logits.argmax(1).cpu())
    all_logits = torch.cat(all_logits, 0)
    preds      = torch.cat(preds_list, 0)
    loss = ce_no_ls(all_logits, ly).item()
    acc  = (preds == ly).float().mean().item()
    model.train()
    return loss, acc

def get_weights(epoch, cfg):
    return dict(cls=cfg.lam_cls,
                gen=cfg.lam_gen if epoch >= cfg.warmup_gen else 0.,
                sc =cfg.lam_sc  if epoch >= cfg.warmup_sc  else 0.,
                pseudo=cfg.lam_pseudo, fm=cfg.lam_fm)

swa_start_ep  = int(cfg.gan_epochs * cfg.swa_start)
refresh_count = 1

history = []   # (epoch, d_train, d_val, g, train_acc, val_acc)
best_val_acc  = 0.0
os.makedirs("models/sgan", exist_ok=True)
save_d_best_path   = "models/sgan/gan_d_best.pth"
save_g_best_path   = "models/sgan/gan_g_best.pth"
save_g_ema_best    = "models/sgan/gan_g_ema_best.pth"

ds1        = make_dataset1(lx_train, ly_train, ux, lp_mask, lp_hard)
loader_lab = make_loader(TensorDataset(lx_train, ly_train, soft_labels[:len(lx_train)]), cfg.gan_batch)
loader_ds1 = make_loader(ds1, cfg.gan_batch)
loader_unl = make_loader(TensorDataset(ux, soft_labels[len(lx_train):]), cfg.gan_batch)

print("\n=== Stage 3: GAN ===")

for epoch in range(cfg.gan_epochs):
    G.train(); D.train()
    W = get_weights(epoch, cfg)

    # --- LP refresh ---
    if epoch > 0 and epoch % cfg.pseudo_refresh == 0:
        tqdm.write(f"  [ep {epoch}] LP refresh (refresh#{refresh_count}) ...")
        soft_labels, lp_mask, lp_hard = label_propagation(
            cfg, simclr_model, lx_train, ly_train, ux,
            refresh_count=refresh_count, D_model=D)
        refresh_count += 1
        ds1        = make_dataset1(lx_train, ly_train, ux, lp_mask, lp_hard)
        loader_ds1 = make_loader(ds1, cfg.gan_batch)
        loader_unl = make_loader(TensorDataset(ux, soft_labels[len(lx_train):]), cfg.gan_batch)
        loader_lab = make_loader(TensorDataset(lx_train, ly_train,
                                               soft_labels[:len(lx_train)]), cfg.gan_batch)

    d_losses, g_losses = [], []
    iter_lab = iter(loader_lab); iter_unl = iter(loader_unl)

    step_bar = tqdm(loader_ds1, desc=f"Epoch {epoch+1}", position=0, leave=True, unit="batch")
    n_batches = len(loader_ds1)
    for _bi, (x_ds1, y_ds1) in enumerate(step_bar):
        x_ds1, y_ds1 = x_ds1.to(cfg.device), y_ds1.to(cfg.device)
        B = x_ds1.size(0)

        try: x_l, y_l, soft_l = next(iter_lab)
        except StopIteration:
            iter_lab = iter(loader_lab); x_l, y_l, soft_l = next(iter_lab)
        x_l, y_l, soft_l = x_l.to(cfg.device), y_l.to(cfg.device), soft_l.to(cfg.device)

        try: x_u, soft_u = next(iter_unl)
        except StopIteration:
            iter_unl = iter(loader_unl); x_u, soft_u = next(iter_unl)
        x_u, soft_u = x_u.to(cfg.device), soft_u.to(cfg.device)

        z      = torch.randn(B, cfg.latent_dim, device=cfg.device)
        y_fake = torch.randint(0, cfg.n_classes, (B,), device=cfg.device)
        with torch.no_grad(): x_fake = G(z, y_fake)

        # ADA augment
        x_real_d = ada.apply(x_ds1)
        x_fake_d = ada.apply(x_fake.detach())

        # FixMatch pseudo-label (weak → decide, strong → train)
        with torch.no_grad():
            x_u_weak      = weak_aug(x_u)
            logits_u_weak = D(x_u_weak)[:, :cfg.n_classes]
            pseudo_prob   = F.softmax(logits_u_weak, dim=-1)

        x_cls, ya, yb, lam = get_cls_view(x_l, y_l, epoch, cfg)
        v1 = diff_augment(x_l); v2 = diff_augment(x_l)

        # ===== D update =====
        opt_D.zero_grad(); opt_cls.zero_grad()

        # real loss を一度だけ forward して ADA 更新にも使う
        logits_real = D(x_real_d, y_ds1)
        # ADA: fake チャネル (index n_classes) への実画像 logit でD強さを測定
        ada.update(logits_real[:, cfg.n_classes].detach())
        loss_d_real = ce_fn(logits_real, y_ds1)
        fake_tgt    = torch.full((len(x_fake_d),), cfg.n_classes, dtype=torch.long, device=cfg.device)
        loss_d_fake = ce_fn(D(x_fake_d), fake_tgt)

        x_u_strong  = diff_augment(x_u)
        logits_unl  = D(x_u_strong)[:, :cfg.n_classes]
        conf_mask   = pseudo_prob.max(1).values >= cfg.fixmatch_thresh
        soft_sharp = F.softmax(logits_u_weak / cfg.fixmatch_temp, dim=-1)
        loss_pseudo = (soft_kl_loss(logits_unl[conf_mask], soft_sharp[conf_mask])
                       if conf_mask.any() else torch.tensor(0., device=cfg.device))

        loss_sc = torch.tensor(0., device=cfg.device)
        if W["sc"] > 0:
            f1 = D.features(v1); f2 = D.features(v2)
            loss_sc = supcon(torch.stack([f1, f2], 1), y_l)

        logits_cls = D(x_cls.to(cfg.device))[:, :cfg.n_classes]
        loss_cls   = (mix_criterion(ce_fn, logits_cls, ya, yb, lam)
                      if lam < 1. else ce_fn(logits_cls, ya))

        loss_D = (W["cls"] * loss_cls + loss_d_real + loss_d_fake +
                  W["pseudo"] * loss_pseudo + W["sc"] * loss_sc)
        loss_D.backward()
        nn.utils.clip_grad_norm_(D.parameters(), 1.)
        opt_D.step(); opt_cls.step()
        d_losses.append(loss_D.item())

        # ===== G update =====
        if W["gen"] > 0:
            opt_G.zero_grad()
            z   = torch.randn(B, cfg.latent_dim, device=cfg.device)
            y_g = torch.randint(0, cfg.n_classes, (B,), device=cfg.device)
            x_gen   = G(z, y_g); x_gen_d = ada.apply(x_gen)
            # hinge loss: Generator は D の fake チャネルを下げる方向に学習
            loss_g_adv = hinge.g(D(x_gen_d, y_g)[:, cfg.n_classes])
            with torch.no_grad(): f_real = D.features(x_ds1[:B])
            loss_fm = feature_matching_loss(f_real, D.features(x_gen))
            loss_G  = W["gen"] * loss_g_adv + W["fm"] * loss_fm
            loss_G.backward()
            nn.utils.clip_grad_norm_(G.parameters(), 1.)
            opt_G.step()
            g_losses.append(loss_G.item())
            update_ema(G, G_ema, cfg.ema_decay)

        step_bar.set_postfix(loss=f"{loss_D.item():.4f}")

    sched_D.step(); sched_G.step(); sched_cls.step()
    if epoch >= swa_start_ep: D_swa.update_parameters(D)

    if (epoch + 1) % cfg.log_interval == 0:
        d_train = np.mean(d_losses) if d_losses else 0.
        g_mean  = np.mean(g_losses) if g_losses else 0.
        d_val_loss, val_acc   = eval_metrics(D, lx_val,   ly_val,   cfg)
        _,          train_acc = eval_metrics(D, lx_train, ly_train, cfg)

        tqdm.write(
            f"[GAN ep {epoch+1:03d}/{cfg.gan_epochs}] "
            f"D_train={d_train:.4f}  D_val={d_val_loss:.4f} | "
            f"train_acc={train_acc*100:.1f}%  val_acc={val_acc*100:.1f}% | "
            f"G={g_mean:.4f}  ada_p={ada.p:.3f}  "
            f"W_gen={W['gen']:.1f}  W_sc={W['sc']:.1f}"
        )
        history.append((epoch + 1, d_train, d_val_loss, g_mean, train_acc, val_acc))

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(D.state_dict(),     save_d_best_path)
            torch.save(G.state_dict(),     save_g_best_path)
            torch.save(G_ema.state_dict(), save_g_ema_best)
            tqdm.write(f"  => [Best] val_acc={best_val_acc*100:.2f}%  saved.")

torch.save(D.state_dict(),     "models/sgan/gan_d_final.pth")
torch.save(G.state_dict(),     "models/sgan/gan_g_final.pth")
torch.save(G_ema.state_dict(), "models/sgan/gan_g_ema_final.pth")
# SWA BN update
tqdm.write("\nSWA BN update ...")
_bn_loader = DataLoader(TensorDataset(torch.cat([lx_train, ux], 0)),
                        batch_size=256, shuffle=True)
torch.optim.swa_utils.update_bn(
    (x.to(cfg.device) for (x,) in tqdm(_bn_loader, desc="SWA BN", position=0, leave=True)),
    D_swa)
torch.save(D_swa.state_dict(), "models/sgan/gan_d_swa_final.pth")
tqdm.write("[SWA] D_swa saved → 'models/sgan/gan_d_swa_final.pth'")



=== Stage 3: GAN ===


Epoch 10: 100%|██████████| 4/4 [00:04<00:00,  1.19s/batch, loss=2.8738]


[GAN ep 010/300] D_train=2.9037  D_val=1.5007 | train_acc=49.8%  val_acc=49.0% | G=0.0000  ada_p=0.040  W_gen=0.0  W_sc=0.0
  => [Best] val_acc=49.00%  saved.


Epoch 20: 100%|██████████| 4/4 [00:05<00:00,  1.26s/batch, loss=2.4691]


[GAN ep 020/300] D_train=2.5394  D_val=0.8643 | train_acc=83.0%  val_acc=80.0% | G=0.0000  ada_p=0.080  W_gen=0.0  W_sc=0.0
  => [Best] val_acc=80.00%  saved.
  [ep 20] LP refresh (refresh#1) ...
  [LP] refresh#1  α=0.92  topk=22/class  conf_floor=0.7
  [LP] confident unlabeled: 220/3000  total (incl. labeled): 620/3400


...

Epoch 90: 100%|██████████| 5/5 [01:34<00:00, 18.86s/batch, loss=3.1397]


[GAN ep 090/300] D_train=3.0709  D_val=0.8043 | train_acc=75.2%  val_acc=69.0% | G=-1.7169  ada_p=0.408  W_gen=0.5  W_sc=0.3


Epoch 100: 100%|██████████| 5/5 [01:39<00:00, 19.95s/batch, loss=2.9549]


[GAN ep 100/300] D_train=2.9533  D_val=0.6393 | train_acc=82.5%  val_acc=77.0% | G=-1.6652  ada_p=0.460  W_gen=0.5  W_sc=0.3
  [ep 100] LP refresh (refresh#5) ...
  [LP] refresh#5  α=0.92  topk=30/class  conf_floor=0.7
  [LP] confident unlabeled: 300/3000  total (incl. labeled): 700/3400


Epoch 110:  80%|████████  | 4/5 [10:41<03:07, 187.74s/batch, loss=4.0432]

In [ ]:
def tta_predict(model, x, n_views=5):
    """
    TTA: 複数 augmentation で予測を平均。
    lambda の closure バグ回避のため関数リストを明示的に定義。
    """
    model.eval()
    def _identity(x):  return x
    def _weak(x):      return weak_aug(x)
    def _strong(x):    return diff_augment(x)
    def _both(x):      return diff_augment(weak_aug(x))
    def _strong2(x):   return diff_augment(x)
    aug_fns = [_identity, _weak, _strong, _both, _strong2]
    probs = []
    with torch.no_grad():
        for fn in aug_fns[:n_views]:
            xv = fn(x.to(DEVICE))
            probs.append(F.softmax(model(xv)[:, :cfg.n_classes], dim=-1))
    return torch.stack(probs).mean(0)

def evaluate_test(model, test_imgs, test_lbls, batch=256, label=""):
    preds = []
    for i in tqdm(range(0, len(test_imgs), batch), desc=f"TTA [{label}]", position=0, leave=True):
        preds.append(tta_predict(model, test_imgs[i:i+batch]).argmax(1).cpu())
    preds = torch.cat(preds)
    acc   = (preds == test_lbls).float().mean().item()
    print(f"[{label}] ★ TEST Accuracy (final): {acc*100:.2f}%")
    return acc

print("\n=== Final Test Evaluation (1回のみ) ===")
if history:
    best_ep = max(history, key=lambda h: h[5])
    print(f"  Best val_acc: {best_ep[5]*100:.1f}%  @ epoch {best_ep[0]}")
evaluate_test(D,     test_imgs, test_lbls, label="D (no SWA)")
evaluate_test(D_swa, test_imgs, test_lbls, label="D_swa")

# 学習曲線プロット (4指標)
try:
    import matplotlib.pyplot as plt
    if history:
        eps        = [h[0] for h in history]
        d_trains   = [h[1] for h in history]
        d_vals     = [h[2] for h in history]
        train_accs = [h[4]*100 for h in history]
        val_accs   = [h[5]*100 for h in history]

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        axes[0].plot(eps, d_trains, label="D train loss", marker="o", ms=3)
        axes[0].plot(eps, d_vals,   label="D val loss",   marker="s", ms=3)
        axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
        axes[0].set_title("Stage 3 — D Loss (train vs val)")
        axes[0].legend(); axes[0].grid(alpha=0.3)

        axes[1].plot(eps, train_accs, label="train acc", marker="o", ms=3)
        axes[1].plot(eps, val_accs,   label="val acc",   marker="s", ms=3)
        axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
        axes[1].set_title("Stage 3 — Accuracy (train vs val)")
        axes[1].legend(); axes[1].grid(alpha=0.3)

        plt.tight_layout(); plt.show()
except ImportError:
    pass

# EMA-G サンプル
G_ema.eval()
with torch.no_grad():
    gs = G_ema(torch.randn(10, cfg.latent_dim, device=DEVICE),
               torch.arange(10, device=DEVICE)).cpu()
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 10, figsize=(20, 2))
    for i, ax in enumerate(axes):
        ax.imshow(gs[i].permute(1, 2, 0).numpy(), cmap="gray")
        ax.set_title(str(i)); ax.axis("off")
    plt.suptitle("EMA-G samples", y=1.02); plt.tight_layout(); plt.show()
except ImportError:
    pass
